# 01 — ARD hello-world

Confirms the GEE Cloud project is wired up and Sentinel-2 / Sentinel-1 collections return real images over Ghana for the 2024 major maize season (Apr–Jul).

Run top-to-bottom. If both image counts are non-zero and the RGB map renders, the pipeline plumbing works.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.ee_init import init, ghana_boundary
from src.ard import s2_collection, add_s2_indices, s1_collection

init()
print('Earth Engine initialized')

In [ ]:
ghana = ghana_boundary()

# Major maize season for southern + transition zones, 2024
START, END = '2024-04-01', '2024-07-31'

s2 = s2_collection(ghana, START, END)
print('Sentinel-2 scenes:', s2.size().getInfo())

s1 = s1_collection(ghana, START, END)
print('Sentinel-1 scenes:', s1.size().getInfo())

In [ ]:
# Median composite + NDVI
composite = s2.map(add_s2_indices).median().clip(ghana)
print('Composite bands:', composite.bandNames().getInfo())

In [ ]:
import geemap

Map = geemap.Map(center=[7.95, -1.03], zoom=6)
Map.addLayer(composite, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'S2 RGB')
Map.addLayer(composite.select('NDVI'), {'min': 0, 'max': 0.9, 'palette': ['white', 'green']}, 'NDVI')
Map.addLayer(ghana.style(**{'color': 'red', 'fillColor': '00000000', 'width': 2}), {}, 'Ghana boundary')
Map